In [1]:
import datetime as dt
import pandas as pd
import pyreadr
import numpy as np
from datetime import datetime, timedelta

import warnings
warnings.filterwarnings("ignore")

import gc
gc.collect()
gc.collect()
gc.collect()

0

In [2]:
df = pd.read_excel(r"D:\TASK\HL & LAP\HL\HL files\HL DISBURSEMENT MOM\9. H.L Active Dump - Dec 2025.xlsx", #engine='pyxlsb',
                  usecols=['AGREEMENTNO', 'LOAN BAND', 'IRR BAND', 'CHANNEL TYPE', 'BRANCHDESC',
       'LOCATION', 'AREA', 'REGION', 'ZONE', 'LTV BAND', 'PROPERTY_DESC','CONSTITUTION_DESC', 'INDUSTRYDESC', 'SUB INDUSTRYDESC', 'AGE BAND',
       'Tenure Slab', 'End Use 1', 'Product_type', 'PRODUCT1', 'SCHEMEDESC',
       'AGR_AUTH_DATE', 'STATUS_DESC','Current WIRR','BUSINESS_IRR', 'DISB_STATUS','AMTFIN'])

df.head()

,AGREEMENTNO,AMTFIN,LOAN BAND,BUSINESS_IRR,IRR BAND,AGR_AUTH_DATE,CHANNEL TYPE,BRANCHDESC,LOCATION,AREA,...,CONSTITUTION_DESC,INDUSTRYDESC,SUB INDUSTRYDESC,AGE BAND,Current WIRR,DISB_STATUS,PRODUCT1,Tenure Slab,End Use 1,Product_type
0,X0HLNUG00000851444,800000.0,<15L,0.15,<13.50%,2012-11-16,Direct,NUNGAMBAKKAM HL,NUNGAMBAKKAM HL,Chennai Area 1,...,AFFORDABLE,Telecommunication/Broadcasting,Service,46-50,120000.0000,FULLY DISBURSED,HL,11-15 Yrs,RESALE,SECURITIZATION FOR HL
1,X0HLCIO00000872328,1200000.0,<15L,0.23,15.50%-16.50%,2012-12-24,Direct,COIMBATORE HL,COIMBATORE HL,Coimbatore Area,...,AFFORDABLE,Jewellery,0,46-50,276000.0000,FULLY DISBURSED,HL,11-15 Yrs,LEAP CASES,HL HOME LOAN
2,X0HLNUG00000881454,1450000.0,<15L,0.23,15.50%-16.50%,2012-12-31,Direct,NUNGAMBAKKAM HL,NUNGAMBAKKAM HL,Chennai Area 1,...,AFFORDABLE,General Merchandise Store,Service,56-60,333500.0000,FULLY DISBURSED,HL,11-15 Yrs,LEAP CASES,HL HOME LOAN
3,X0HLNUG00000883612,2000000.0,15L-25L,0.23,15.50%-16.50%,2012-12-31,Direct,NUNGAMBAKKAM HL,NUNGAMBAKKAM HL,Chennai Area 1,...,AFFORDABLE,Computer & related products,Job Work,46-50,460000.0000,FULLY DISBURSED,HL,11-15 Yrs,LEAP CASES,HL HOME LOAN
4,X0HLKAC00000881532,1000000.0,<15L,0.23,>17.50%,2012-12-31,Direct,KANCHIPURAM HL,KANCHIPURAM HL,Kancheepuram Area,...,AFFORDABLE,Dairy Farming,Whole Sale and Retail Trading,65-70,210813.5196,FULLY DISBURSED,HL,11-15 Yrs,SELF CONSTRUCTION,HL HOME LOAN


In [3]:
df['AGR_AUTH_DATE'] = pd.to_datetime(df['AGR_AUTH_DATE']) 
df = df.loc[(df.AGR_AUTH_DATE >= "2023-04-01")]              # can be changed based on req
df = df.loc[df.STATUS_DESC.isin(['ACTIVE', 'CLOSED'])]
print(df.shape)
df.head(3)

(137003, 26)


,AGREEMENTNO,AMTFIN,LOAN BAND,BUSINESS_IRR,IRR BAND,AGR_AUTH_DATE,CHANNEL TYPE,BRANCHDESC,LOCATION,AREA,...,CONSTITUTION_DESC,INDUSTRYDESC,SUB INDUSTRYDESC,AGE BAND,Current WIRR,DISB_STATUS,PRODUCT1,Tenure Slab,End Use 1,Product_type
89885,LAP1ALL000074445,1300000.0,<15L,0.1375,15.50%-16.50%,2023-04-03,DSA,ALLEPPEY HL,ALLEPPEY HL,Alleppey Area,...,AFFORDABLE,NaN,NaN,46-50,178750.0000,FULLY DISBURSED,LAP,11-15 Yrs,LAP,LAP-RESALE INDEPENDENT HOUSE
89886,LAP1ALL000074449,1352351.0,<15L,0.1375,15.50%-16.50%,2023-04-03,CRA,ALLEPPEY HL,ALLEPPEY HL,Alleppey Area,...,AFFORDABLE,NaN,NaN,56-60,185948.2625,FULLY DISBURSED,LAP,11-15 Yrs,LAP,LAP-RESALE INDEPENDENT HOUSE
89887,LAP1ALL000074450,650000.0,<15L,0.1375,15.50%-16.50%,2023-04-03,DSA,ALLEPPEY HL,ALLEPPEY HL,Alleppey Area,...,AFFORDABLE,NaN,NaN,<35,89375.0000,FULLY DISBURSED,LAP,11-15 Yrs,LAP,LAP-RESALE INDEPENDENT HOUSE


In [4]:
dl = pd.read_csv(r"D:\TASK\HE_HL_allcols_dump_Rcode\output HE & HL\HE_HL_deldump_cash_flow_mar21_dec25_AllCol.csv", encoding = "latin1")
base_month = dl[["AGREEMENTNO","INTCOMP_BILLED","INTCOMP_RECD","UNADJUST","EMI_OS","POS","DISBURSEDAMT","EMI_STARTDATE","DPD","OD_PLUS_POS","CUSTOMERNAME","month"]]
#final = final.merge(dl.loc[dl.month == "may-2024"][["AGREEMENTNO","EMI_STARTDATE"]], on="AGREEMENTNO", how="left")


###



EMI_STARTDATE Mapping

In [5]:
# Taking recent month emi date and mapping against agreementno's

dl['EMI_STARTDATE']= pd.to_datetime(dl['EMI_STARTDATE'],  errors='coerce')
dl['EMI_STARTDATE']= dl['EMI_STARTDATE'].dt.strftime('%Y-%m-%d')

emi1= dl.loc[(dl['month']=='dec-2025'), ['AGREEMENTNO','EMI_STARTDATE']]
rough1= emi1.merge(df['AGREEMENTNO'], on='AGREEMENTNO', how='right')
print(rough1.shape)
rough1.head(3)

(137003, 2)


,AGREEMENTNO,EMI_STARTDATE
0,LAP1ALL000074445,2023-05-06
1,LAP1ALL000074449,2023-05-06
2,LAP1ALL000074450,2023-05-06


In [6]:
# mapping the emi_start date for closed cases agreements

emi2 = dl.drop_duplicates(subset='AGREEMENTNO', keep='last')
emi2= emi2[['AGREEMENTNO','EMI_STARTDATE']]
emi2.shape

(405550, 2)

In [7]:
rough2= rough1.merge(emi2, on='AGREEMENTNO', how='left')
print(rough2.shape)
rough2.head(3)

(137003, 3)


,AGREEMENTNO,EMI_STARTDATE_x,EMI_STARTDATE_y
0,LAP1ALL000074445,2023-05-06,NaN
1,LAP1ALL000074449,2023-05-06,NaN
2,LAP1ALL000074450,2023-05-06,NaN


In [8]:
rough2.isna().sum()

AGREEMENTNO             0
EMI_STARTDATE_x     26221
EMI_STARTDATE_y    130275
dtype: int64

In [9]:
rough2['EMI_STARTDATE_x'] = rough2.apply(lambda row: row['EMI_STARTDATE_y'] if pd.isnull(row['EMI_STARTDATE_x']) else row['EMI_STARTDATE_x'], axis=1)
rough2.isna().sum()

AGREEMENTNO             0
EMI_STARTDATE_x     22845
EMI_STARTDATE_y    130275
dtype: int64

In [10]:
emi= rough2[['AGREEMENTNO','EMI_STARTDATE_x']]
col={'EMI_STARTDATE_x':'EMI_STARTDATE'}
emi= emi.rename(columns=col)
emi.head(3)

,AGREEMENTNO,EMI_STARTDATE
0,LAP1ALL000074445,2023-05-06
1,LAP1ALL000074449,2023-05-06
2,LAP1ALL000074450,2023-05-06


In [11]:
data= df.merge(emi, how='left', on='AGREEMENTNO')
data.shape

(137003, 27)

In [12]:
# mapping the disb_status column

dfs2 = dl.loc[(dl['month']=='dec-2025'), ['AGREEMENTNO','DISB_STATUS']]
data= data.merge(dfs2[['AGREEMENTNO','DISB_STATUS']], on='AGREEMENTNO', how='left')

In [13]:
dfs1 = dl.drop_duplicates(subset='AGREEMENTNO', keep='last')
data = data.merge(dfs1[['AGREEMENTNO', 'DISB_STATUS']],  on='AGREEMENTNO', how='left')
data['DISB_STATUS_x'] = data.apply(lambda row: row['DISB_STATUS_y'] if pd.isnull(row['DISB_STATUS_x']) else row['DISB_STATUS_x'], axis=1)
data= data.drop('DISB_STATUS_y', axis=1)
col={'DISB_STATUS_x':'DISBS_STATUS'}
data= data.rename(columns=col)
data.head(3)

,AGREEMENTNO,AMTFIN,LOAN BAND,BUSINESS_IRR,IRR BAND,AGR_AUTH_DATE,CHANNEL TYPE,BRANCHDESC,LOCATION,AREA,...,SUB INDUSTRYDESC,AGE BAND,Current WIRR,DISBS_STATUS,PRODUCT1,Tenure Slab,End Use 1,Product_type,EMI_STARTDATE,DISB_STATUS
0,LAP1ALL000074445,1300000.0,<15L,0.1375,15.50%-16.50%,2023-04-03,DSA,ALLEPPEY HL,ALLEPPEY HL,Alleppey Area,...,NaN,46-50,178750.0000,FULLY DISBURSED,LAP,11-15 Yrs,LAP,LAP-RESALE INDEPENDENT HOUSE,2023-05-06,FULLY DISBURSED
1,LAP1ALL000074449,1352351.0,<15L,0.1375,15.50%-16.50%,2023-04-03,CRA,ALLEPPEY HL,ALLEPPEY HL,Alleppey Area,...,NaN,56-60,185948.2625,FULLY DISBURSED,LAP,11-15 Yrs,LAP,LAP-RESALE INDEPENDENT HOUSE,2023-05-06,FULLY DISBURSED
2,LAP1ALL000074450,650000.0,<15L,0.1375,15.50%-16.50%,2023-04-03,DSA,ALLEPPEY HL,ALLEPPEY HL,Alleppey Area,...,NaN,<35,89375.0000,FULLY DISBURSED,LAP,11-15 Yrs,LAP,LAP-RESALE INDEPENDENT HOUSE,2023-05-06,FULLY DISBURSED


In [14]:
data.isnull().sum()

AGREEMENTNO              0
AMTFIN                   0
LOAN BAND                0
BUSINESS_IRR             0
IRR BAND                 0
AGR_AUTH_DATE            0
CHANNEL TYPE             0
BRANCHDESC               0
LOCATION                 0
AREA                     0
REGION                   0
ZONE                     0
STATUS_DESC              0
SCHEMEDESC               0
LTV BAND                 0
PROPERTY_DESC            0
CONSTITUTION_DESC        0
INDUSTRYDESC          9116
SUB INDUSTRYDESC     10858
AGE BAND                 0
Current WIRR             0
DISBS_STATUS             0
PRODUCT1                 0
Tenure Slab              0
End Use 1                0
Product_type             0
EMI_STARTDATE        22845
DISB_STATUS              0
dtype: int64

In [15]:
data['DISBS_STATUS'].nunique()

2

## Work on the EMI_STARTDATE for partially disbursed cases & null

In [16]:
data['AGR_AUTH_DATE']= pd.to_datetime(data['AGR_AUTH_DATE'], format="%Y-%m-%d")
data['EMI_STARTDATE']= pd.to_datetime(data['EMI_STARTDATE'], format="%Y-%d-%m").dt.strftime('%Y-%m-%d')

In [17]:
## Code to create a one month ahead from agr_auth_date

data['AGR_AUTH_DATE']= pd.to_datetime(data['AGR_AUTH_DATE'], format="%Y-%m-%d")

def calculate_one_month_ahead(date):
    next_month = date + timedelta(days=30)
    next_month_5th = datetime(next_month.year, next_month.month, 5)
    return next_month_5th

data['one_month_ahead_flag'] = data['AGR_AUTH_DATE'].apply(calculate_one_month_ahead)

#data['one_month_ahead_flag'] = data['AGR_AUTH_DATE']+ pd.DateOffset(months=1)

data.head(3)

,AGREEMENTNO,AMTFIN,LOAN BAND,BUSINESS_IRR,IRR BAND,AGR_AUTH_DATE,CHANNEL TYPE,BRANCHDESC,LOCATION,AREA,...,AGE BAND,Current WIRR,DISBS_STATUS,PRODUCT1,Tenure Slab,End Use 1,Product_type,EMI_STARTDATE,DISB_STATUS,one_month_ahead_flag
0,LAP1ALL000074445,1300000.0,<15L,0.1375,15.50%-16.50%,2023-04-03,DSA,ALLEPPEY HL,ALLEPPEY HL,Alleppey Area,...,46-50,178750.0000,FULLY DISBURSED,LAP,11-15 Yrs,LAP,LAP-RESALE INDEPENDENT HOUSE,2023-06-05,FULLY DISBURSED,2023-05-05
1,LAP1ALL000074449,1352351.0,<15L,0.1375,15.50%-16.50%,2023-04-03,CRA,ALLEPPEY HL,ALLEPPEY HL,Alleppey Area,...,56-60,185948.2625,FULLY DISBURSED,LAP,11-15 Yrs,LAP,LAP-RESALE INDEPENDENT HOUSE,2023-06-05,FULLY DISBURSED,2023-05-05
2,LAP1ALL000074450,650000.0,<15L,0.1375,15.50%-16.50%,2023-04-03,DSA,ALLEPPEY HL,ALLEPPEY HL,Alleppey Area,...,<35,89375.0000,FULLY DISBURSED,LAP,11-15 Yrs,LAP,LAP-RESALE INDEPENDENT HOUSE,2023-06-05,FULLY DISBURSED,2023-05-05


In [18]:
# For EMI StartDate null cases and DISB_STATUS 'partially Disbursed' mark one_month_ahead_flag as EMI STARTDATE 

data['EMI_STARTDATE']= pd.to_datetime(data['EMI_STARTDATE'], format="%Y-%m-%d")
data['one_month_ahead_flag']= pd.to_datetime(data['one_month_ahead_flag'], format="%Y-%m-%d")

data['EMI_STARTDATE']= data['EMI_STARTDATE'].fillna(data['one_month_ahead_flag'])


In [19]:
data.isna().sum()

AGREEMENTNO                 0
AMTFIN                      0
LOAN BAND                   0
BUSINESS_IRR                0
IRR BAND                    0
AGR_AUTH_DATE               0
CHANNEL TYPE                0
BRANCHDESC                  0
LOCATION                    0
AREA                        0
REGION                      0
ZONE                        0
STATUS_DESC                 0
SCHEMEDESC                  0
LTV BAND                    0
PROPERTY_DESC               0
CONSTITUTION_DESC           0
INDUSTRYDESC             9116
SUB INDUSTRYDESC        10858
AGE BAND                    0
Current WIRR                0
DISBS_STATUS                0
PRODUCT1                    0
Tenure Slab                 0
End Use 1                   0
Product_type                0
EMI_STARTDATE               0
DISB_STATUS                 0
one_month_ahead_flag        0
dtype: int64

In [20]:
data= data.fillna('Unmapped')

In [21]:
data= data.drop(['one_month_ahead_flag'], axis=1)
data.shape

(137003, 28)

In [22]:
data.drop_duplicates().to_excel(r"D:\TASK\HL & LAP\HL\HL_static Pool\final.xlsx", index=False)
print("saved successfully!")

saved successfully!


In [23]:
#Loading final

pd.set_option('display.max_columns', 100)
final= pd.read_excel(r'D:\TASK\HL & LAP\HL\HL_static Pool\final.xlsx')

In [24]:
final['AGR_AUTH_DATE']= pd.to_datetime(final['AGR_AUTH_DATE'], format="%d/%m/%Y")
final['EMI_STARTDATE']= pd.to_datetime(final['EMI_STARTDATE'], format="%d/%m/%Y")
final.head(5)

,AGREEMENTNO,AMTFIN,LOAN BAND,BUSINESS_IRR,IRR BAND,AGR_AUTH_DATE,CHANNEL TYPE,BRANCHDESC,LOCATION,AREA,REGION,ZONE,STATUS_DESC,SCHEMEDESC,LTV BAND,PROPERTY_DESC,CONSTITUTION_DESC,INDUSTRYDESC,SUB INDUSTRYDESC,AGE BAND,Current WIRR,DISBS_STATUS,PRODUCT1,Tenure Slab,End Use 1,Product_type,EMI_STARTDATE,DISB_STATUS
0,LAP1ALL000074445,1300000,<15L,0.1375,15.50%-16.50%,2023-04-03,DSA,ALLEPPEY HL,ALLEPPEY HL,Alleppey Area,Kerala 2,South 1,ACTIVE,LAP,LT50%,FLOATING,AFFORDABLE,Unmapped,Unmapped,46-50,178750.0000,FULLY DISBURSED,LAP,11-15 Yrs,LAP,LAP-RESALE INDEPENDENT HOUSE,2023-06-05,FULLY DISBURSED
1,LAP1ALL000074449,1352351,<15L,0.1375,15.50%-16.50%,2023-04-03,CRA,ALLEPPEY HL,ALLEPPEY HL,Alleppey Area,Kerala 2,South 1,ACTIVE,LAP,LT50%,FLOATING,AFFORDABLE,Unmapped,Unmapped,56-60,185948.2625,FULLY DISBURSED,LAP,11-15 Yrs,LAP,LAP-RESALE INDEPENDENT HOUSE,2023-06-05,FULLY DISBURSED
2,LAP1ALL000074450,650000,<15L,0.1375,15.50%-16.50%,2023-04-03,DSA,ALLEPPEY HL,ALLEPPEY HL,Alleppey Area,Kerala 2,South 1,ACTIVE,LAP,LT50%,FLOATING,AFFORDABLE,Unmapped,Unmapped,<35,89375.0000,FULLY DISBURSED,LAP,11-15 Yrs,LAP,LAP-RESALE INDEPENDENT HOUSE,2023-06-05,FULLY DISBURSED
3,HL25ALL000074444,2800000,25L-30L,0.0850,13.50%-14.50%,2023-04-03,DSA,ALLEPPEY HL,ALLEPPEY HL,Alleppey Area,Kerala 2,South 1,ACTIVE,HL,70%-80%,FLOATING,AFFORDABLE,Unmapped,Unmapped,<35,238000.0000,FULLY DISBURSED,HL,16-20 Yrs,SELF CONSTRUCTION,RESIDENTIAL SELF CONSTRUCTION,2023-06-05,FULLY DISBURSED
4,LAP1ALL000074448,1083837,<15L,0.1375,15.50%-16.50%,2023-04-03,DSA,THIRUVALLA HL,THIRUVALLA HL,Alleppey Area,Kerala 2,South 1,ACTIVE,LAP,LT50%,FLOATING,AFFORDABLE,Unmapped,Unmapped,46-50,149027.5875,FULLY DISBURSED,LAP,11-15 Yrs,LAP,LAP-RESALE INDEPENDENT HOUSE,2023-06-05,FULLY DISBURSED


### base month file workings

In [25]:
base_month = base_month.drop(['EMI_STARTDATE'], axis=1)
print(base_month.shape)
base_month.head(4)

(8300810, 11)


,AGREEMENTNO,INTCOMP_BILLED,INTCOMP_RECD,UNADJUST,EMI_OS,POS,DISBURSEDAMT,DPD,OD_PLUS_POS,CUSTOMERNAME,month
0,EF01ABA0000067662,52364,52364,0,0.0,825780.0,1212735,NaN,NaN,DAWAT FAMILY RESTAURANT,NaN
1,EF01ABA0000067662,6047,6047,0,0.0,1212735.0,"12,12,735",0.0,1212735.0,DAWAT FAMILY RESTAURANT,aug-2025
2,EF01ABA0000067662,42354,42354,0,0.0,924088.0,1212735,0.0,825780.0,DAWAT FAMILY RESTAURANT,dec-2025
3,EF01ABA0000067662,0,0,0,0.0,1212735.0,"12,12,735",0.0,1212735.0,DAWAT FAMILY RESTAURANT,jul-2025


In [26]:
base_month.isna().sum()

AGREEMENTNO            0
INTCOMP_BILLED         0
INTCOMP_RECD           0
UNADJUST               0
EMI_OS                 0
POS                    0
DISBURSEDAMT           0
DPD               302064
OD_PLUS_POS       302064
CUSTOMERNAME           0
month             297652
dtype: int64

In [27]:
base_month.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8300810 entries, 0 to 8300809
Data columns (total 11 columns):
 #   Column          Dtype  
---  ------          -----  
 0   AGREEMENTNO     object 
 1   INTCOMP_BILLED  object 
 2   INTCOMP_RECD    object 
 3   UNADJUST        object 
 4   EMI_OS          float64
 5   POS             float64
 6   DISBURSEDAMT    object 
 7   DPD             float64
 8   OD_PLUS_POS     float64
 9   CUSTOMERNAME    object 
 10  month           object 
dtypes: float64(4), object(7)
memory usage: 696.6+ MB


In [28]:
base_month['POS'] = pd.to_numeric(base_month['POS'].astype(str).str.replace(',', ''), errors='coerce').fillna(0)
base_month['EMI_OS'] = pd.to_numeric(base_month['EMI_OS'].astype(str).str.replace(',', ''), errors='coerce').fillna(0)
base_month['UNADJUST'] = pd.to_numeric(base_month['UNADJUST'].astype(str).str.replace(',', ''), errors='coerce').fillna(0)
base_month['INTCOMP_BILLED'] = pd.to_numeric(base_month['INTCOMP_BILLED'].astype(str).str.replace(',', ''), errors='coerce').fillna(0)
base_month['INTCOMP_RECD'] = pd.to_numeric(base_month['INTCOMP_RECD'].astype(str).str.replace(',', ''), errors='coerce').fillna(0)
base_month['DISBURSEDAMT'] = pd.to_numeric(base_month['DISBURSEDAMT'].astype(str).str.replace(',', ''), errors='coerce').fillna(0)

In [29]:
# POS + EMI_OS-UNADJUST-INT_COMP_BILLED+INT_COMP_RECD

base_month['POS_N'] = base_month.POS + base_month.EMI_OS - base_month.UNADJUST - base_month.INTCOMP_BILLED + base_month.INTCOMP_RECD

In [30]:
base_month = final.merge(base_month[['AGREEMENTNO', 'DPD', 'POS_N', 'DISBURSEDAMT', 'CUSTOMERNAME', 'month']], on="AGREEMENTNO", how="inner").drop_duplicates()

In [31]:
base_month['month_date']= pd.to_datetime(base_month['month'], format="%b-%Y")
base_month['month_date'] = base_month['month_date'].dt.strftime('%Y-%m-%d')
base_month.head(2)

,AGREEMENTNO,AMTFIN,LOAN BAND,BUSINESS_IRR,IRR BAND,AGR_AUTH_DATE,CHANNEL TYPE,BRANCHDESC,LOCATION,AREA,REGION,ZONE,STATUS_DESC,SCHEMEDESC,LTV BAND,PROPERTY_DESC,CONSTITUTION_DESC,INDUSTRYDESC,SUB INDUSTRYDESC,AGE BAND,Current WIRR,DISBS_STATUS,PRODUCT1,Tenure Slab,End Use 1,Product_type,EMI_STARTDATE,DISB_STATUS,DPD,POS_N,DISBURSEDAMT,CUSTOMERNAME,month,month_date
0,LAP1ALL000074445,1300000,<15L,0.1375,15.50%-16.50%,2023-04-03,DSA,ALLEPPEY HL,ALLEPPEY HL,Alleppey Area,Kerala 2,South 1,ACTIVE,LAP,LT50%,FLOATING,AFFORDABLE,Unmapped,Unmapped,46-50,178750.0,FULLY DISBURSED,LAP,11-15 Yrs,LAP,LAP-RESALE INDEPENDENT HOUSE,2023-06-05,FULLY DISBURSED,NaN,1232973.0,1300000.0,SHAJI S,NaN,NaN
1,LAP1ALL000074445,1300000,<15L,0.1375,15.50%-16.50%,2023-04-03,DSA,ALLEPPEY HL,ALLEPPEY HL,Alleppey Area,Kerala 2,South 1,ACTIVE,LAP,LT50%,FLOATING,AFFORDABLE,Unmapped,Unmapped,46-50,178750.0,FULLY DISBURSED,LAP,11-15 Yrs,LAP,LAP-RESALE INDEPENDENT HOUSE,2023-06-05,FULLY DISBURSED,0.0,1279294.0,1300000.0,SHAJI S,apr-2024,2024-04-01


In [32]:
base_month['month_date'] = pd.to_datetime(base_month['month_date'], format='%Y-%m-%d')
base_month['EMI_STARTDATE'] = pd.to_datetime(base_month['EMI_STARTDATE'], format='%Y-%m-%d')   
base_month['mob'] = ((base_month['month_date'].dt.month - base_month['EMI_STARTDATE'].dt.month)+
                     (base_month['month_date'].dt.year - base_month['EMI_STARTDATE'].dt.year)*12)
base_month.head(3)

,AGREEMENTNO,AMTFIN,LOAN BAND,BUSINESS_IRR,IRR BAND,AGR_AUTH_DATE,CHANNEL TYPE,BRANCHDESC,LOCATION,AREA,REGION,ZONE,STATUS_DESC,SCHEMEDESC,LTV BAND,PROPERTY_DESC,CONSTITUTION_DESC,INDUSTRYDESC,SUB INDUSTRYDESC,AGE BAND,Current WIRR,DISBS_STATUS,PRODUCT1,Tenure Slab,End Use 1,Product_type,EMI_STARTDATE,DISB_STATUS,DPD,POS_N,DISBURSEDAMT,CUSTOMERNAME,month,month_date,mob
0,LAP1ALL000074445,1300000,<15L,0.1375,15.50%-16.50%,2023-04-03,DSA,ALLEPPEY HL,ALLEPPEY HL,Alleppey Area,Kerala 2,South 1,ACTIVE,LAP,LT50%,FLOATING,AFFORDABLE,Unmapped,Unmapped,46-50,178750.0,FULLY DISBURSED,LAP,11-15 Yrs,LAP,LAP-RESALE INDEPENDENT HOUSE,2023-06-05,FULLY DISBURSED,NaN,1232973.0,1300000.0,SHAJI S,NaN,NaT,NaN
1,LAP1ALL000074445,1300000,<15L,0.1375,15.50%-16.50%,2023-04-03,DSA,ALLEPPEY HL,ALLEPPEY HL,Alleppey Area,Kerala 2,South 1,ACTIVE,LAP,LT50%,FLOATING,AFFORDABLE,Unmapped,Unmapped,46-50,178750.0,FULLY DISBURSED,LAP,11-15 Yrs,LAP,LAP-RESALE INDEPENDENT HOUSE,2023-06-05,FULLY DISBURSED,0.0,1279294.0,1300000.0,SHAJI S,apr-2024,2024-04-01,10.0
2,LAP1ALL000074445,1300000,<15L,0.1375,15.50%-16.50%,2023-04-03,DSA,ALLEPPEY HL,ALLEPPEY HL,Alleppey Area,Kerala 2,South 1,ACTIVE,LAP,LT50%,FLOATING,AFFORDABLE,Unmapped,Unmapped,46-50,178750.0,FULLY DISBURSED,LAP,11-15 Yrs,LAP,LAP-RESALE INDEPENDENT HOUSE,2023-06-05,FULLY DISBURSED,0.0,1252984.0,1300000.0,SHAJI S,apr-2025,2025-04-01,22.0


In [33]:
base_month.isna().sum()

AGREEMENTNO               0
AMTFIN                    0
LOAN BAND                 0
BUSINESS_IRR              0
IRR BAND                  0
AGR_AUTH_DATE             0
CHANNEL TYPE              0
BRANCHDESC                0
LOCATION                  0
AREA                      0
REGION                    0
ZONE                      0
STATUS_DESC               0
SCHEMEDESC                0
LTV BAND                  0
PROPERTY_DESC             0
CONSTITUTION_DESC         0
INDUSTRYDESC              0
SUB INDUSTRYDESC          0
AGE BAND                  0
Current WIRR              0
DISBS_STATUS              0
PRODUCT1                  0
Tenure Slab               0
End Use 1                 0
Product_type              0
EMI_STARTDATE             0
DISB_STATUS               0
DPD                  129327
POS_N                     0
DISBURSEDAMT              0
CUSTOMERNAME              0
month                128537
month_date           128537
mob                  128537
dtype: int64

In [34]:
base_month['mob_order'] = base_month['mob']
base_month['mob'] = base_month['mob_order'].astype(str) + "M"

In [35]:
def dpd_stage(dpd):
    if dpd > 90:
        return "stage 3"
    elif 30 < dpd <= 90:
        return "stage 2"
    else:
        return "stage 1"
    
base_month['Value'] = base_month['DPD'].apply(dpd_stage)

In [36]:
base_month = base_month.loc[base_month.mob_order >=0 ]
print(base_month.shape)

(1681388, 37)


In [37]:
base_month['mob_order'].unique()

array([10., 22.,  2., 14., 26.,  6., 18., 30.,  8., 20.,  7., 19.,  1.,
       13., 25.,  0., 12., 24.,  9., 21., 11., 23.,  5., 17.,  4., 16.,
       28.,  3., 15., 27., 31., 29.])

In [38]:
max=base_month['mob_order'].max()
print(max)

31.0


In [39]:
# Saving base month file
base_month.drop_duplicates().to_csv(r"D:\TASK\HL & LAP\HL\HL_static Pool\base_month.csv", index=False)


In [40]:
#base_month = base_month.loc[base_month.mob_order >=0 ]
#base_month.drop_duplicates().to_csv("E:\\Abhishek\\code\\HL pool\\most updated HLSPA wrt BM.csv", index = False)

#### Static Pool workings

In [40]:
# Read CSV files
final = pd.read_excel(r"D:\TASK\HL & LAP\HL\HL_static Pool\final.xlsx").drop_duplicates()
final['AGR_AUTH_DATE']= pd.to_datetime(data['AGR_AUTH_DATE'], format="%d/%m/%Y")
final['EMI_STARTDATE']= pd.to_datetime(data['EMI_STARTDATE'], format="%d/%m/%Y")

base_month = pd.read_csv(r"D:\TASK\HL & LAP\HL\HL_static Pool\base_month.csv", 
                         usecols = ["AGREEMENTNO","DPD","mob_order","month"])
base_month = base_month.fillna(0)

# Define DPD stages mapping function
def dpd_stage(dpd):
    if dpd > 90:
        return "stage 3"
    elif 30 < dpd <= 90:
        return "stage 2"
    else:
        return "stage 1"

max=int(base_month['mob_order'].max())

# Process base_month data and merge with final
for m in range(max+1):
    mob = base_month[base_month['mob_order'] == m].copy()
    mob['DPD'] = mob['DPD'].astype(int)
    mob[str(m) + 'M_curr'] = mob['DPD'].apply(dpd_stage)
#     mob.rename(columns={'OD_PLUS_POS': str(m) + 'M_gv'}, inplace=True)

#     final = final.merge(mob[['AGREEMENTNO', str(m) + 'M_curr', str(m) + 'M_gv']], on='AGREEMENTNO', how='left')
    final = final.merge(mob[['AGREEMENTNO', str(m) + 'M_curr']], on='AGREEMENTNO', how='left')
    final = final.drop_duplicates()
    print(m)

0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31


In [41]:
final.AGR_AUTH_DATE.max()

Timestamp('2025-12-31 00:00:00')

In [42]:
final.AGR_AUTH_DATE.min()

Timestamp('2023-04-03 00:00:00')

### Melting final to build LAP SPA base file

In [43]:
df = final.drop_duplicates()
df.columns

Index(['AGREEMENTNO', 'AMTFIN', 'LOAN BAND', 'BUSINESS_IRR', 'IRR BAND',
       'AGR_AUTH_DATE', 'CHANNEL TYPE', 'BRANCHDESC', 'LOCATION', 'AREA',
       'REGION', 'ZONE', 'STATUS_DESC', 'SCHEMEDESC', 'LTV BAND',
       'PROPERTY_DESC', 'CONSTITUTION_DESC', 'INDUSTRYDESC',
       'SUB INDUSTRYDESC', 'AGE BAND', 'Current WIRR', 'DISBS_STATUS',
       'PRODUCT1', 'Tenure Slab', 'End Use 1', 'Product_type', 'EMI_STARTDATE',
       'DISB_STATUS', '0M_curr', '1M_curr', '2M_curr', '3M_curr', '4M_curr',
       '5M_curr', '6M_curr', '7M_curr', '8M_curr', '9M_curr', '10M_curr',
       '11M_curr', '12M_curr', '13M_curr', '14M_curr', '15M_curr', '16M_curr',
       '17M_curr', '18M_curr', '19M_curr', '20M_curr', '21M_curr', '22M_curr',
       '23M_curr', '24M_curr', '25M_curr', '26M_curr', '27M_curr', '28M_curr',
       '29M_curr', '30M_curr', '31M_curr'],
      dtype='object')

In [44]:
import re

id_vars=['AGREEMENTNO', 'AMTFIN', 'LOAN BAND', 'BUSINESS_IRR', 'IRR BAND',
       'AGR_AUTH_DATE', 'CHANNEL TYPE', 'BRANCHDESC', 'LOCATION', 'AREA',
       'REGION', 'ZONE', 'STATUS_DESC', 'SCHEMEDESC', 'LTV BAND',
       'PROPERTY_DESC', 'CONSTITUTION_DESC', 'INDUSTRYDESC',
       'SUB INDUSTRYDESC', 'AGE BAND', 'Current WIRR', 'DISBS_STATUS',
        'PRODUCT1', 'Tenure Slab', 'End Use 1', 'Product_type', 'EMI_STARTDATE','DISB_STATUS']

pattern =  re.compile(r'^\d+M_curr$')
value_var = [c for c in df.columns if pattern.match(c)]

# Sort value_vars numerically by the leading number
def month_num(col):
    return int(col.split('M_')[0])

value_vars= sorted(value_var, key=month_num)
value_vars

['0M_curr',
 '1M_curr',
 '2M_curr',
 '3M_curr',
 '4M_curr',
 '5M_curr',
 '6M_curr',
 '7M_curr',
 '8M_curr',
 '9M_curr',
 '10M_curr',
 '11M_curr',
 '12M_curr',
 '13M_curr',
 '14M_curr',
 '15M_curr',
 '16M_curr',
 '17M_curr',
 '18M_curr',
 '19M_curr',
 '20M_curr',
 '21M_curr',
 '22M_curr',
 '23M_curr',
 '24M_curr',
 '25M_curr',
 '26M_curr',
 '27M_curr',
 '28M_curr',
 '29M_curr',
 '30M_curr',
 '31M_curr']

In [45]:
## Keep adding the M_curr in the value_vars=[ ] of code as replicating in the columns name. like mentioned below in image.

df = pd.melt(df, id_vars= id_vars, 
        value_vars= value_vars, var_name='Attribute', value_name='Value')

df['mob_order'] = pd.to_numeric(df['Attribute'].str.extract(r'([\d.]+)M_curr')[0])
df['mob'] = df['mob_order'].astype(str) + "M"
df["conc"] = df[["AGREEMENTNO", "mob"]].apply(" ".join, axis=1)

In [46]:
pd.unique(df.Value)

array(['stage 1', nan, 'stage 3', 'stage 2'], dtype=object)

In [47]:
df.columns

Index(['AGREEMENTNO', 'AMTFIN', 'LOAN BAND', 'BUSINESS_IRR', 'IRR BAND',
       'AGR_AUTH_DATE', 'CHANNEL TYPE', 'BRANCHDESC', 'LOCATION', 'AREA',
       'REGION', 'ZONE', 'STATUS_DESC', 'SCHEMEDESC', 'LTV BAND',
       'PROPERTY_DESC', 'CONSTITUTION_DESC', 'INDUSTRYDESC',
       'SUB INDUSTRYDESC', 'AGE BAND', 'Current WIRR', 'DISBS_STATUS',
       'PRODUCT1', 'Tenure Slab', 'End Use 1', 'Product_type', 'EMI_STARTDATE',
       'DISB_STATUS', 'Attribute', 'Value', 'mob_order', 'mob', 'conc'],
      dtype='object')

In [48]:
# Taking POS_N, DisbursedAMt, PRODUCT, SCHEME, DISB_STATUS cols from base month file

base= pd.read_csv(r"D:\TASK\HL & LAP\HL\HL_static Pool\base_month.csv", usecols=['DPD','POS_N', 'AGREEMENTNO', 'DISBURSEDAMT', 'CUSTOMERNAME','mob', 'month'])
base["conc"] = base[["AGREEMENTNO", "mob"]].apply(" ".join, axis=1)
base.columns

Index(['AGREEMENTNO', 'DPD', 'POS_N', 'DISBURSEDAMT', 'CUSTOMERNAME', 'month',
       'mob', 'conc'],
      dtype='object')

In [49]:
# Doing the left join of few cols from base file with melted file

new = df.merge(base[[ 'POS_N', 'DISBURSEDAMT', 'DPD', 'CUSTOMERNAME', 'month', 'conc']], on="conc", how = "left")

In [50]:
new.loc[pd.isnull(new.DISBURSEDAMT), "DISBURSEDAMT"] = new.AMTFIN
new.head(3)

,AGREEMENTNO,AMTFIN,LOAN BAND,BUSINESS_IRR,IRR BAND,AGR_AUTH_DATE,CHANNEL TYPE,BRANCHDESC,LOCATION,AREA,REGION,ZONE,STATUS_DESC,SCHEMEDESC,LTV BAND,PROPERTY_DESC,CONSTITUTION_DESC,INDUSTRYDESC,SUB INDUSTRYDESC,AGE BAND,Current WIRR,DISBS_STATUS,PRODUCT1,Tenure Slab,End Use 1,Product_type,EMI_STARTDATE,DISB_STATUS,Attribute,Value,mob_order,mob,conc,POS_N,DISBURSEDAMT,DPD,CUSTOMERNAME,month
0,LAP1ALL000074445,1300000,<15L,0.1375,15.50%-16.50%,2023-04-03,DSA,ALLEPPEY HL,ALLEPPEY HL,Alleppey Area,Kerala 2,South 1,ACTIVE,LAP,LT50%,FLOATING,AFFORDABLE,Unmapped,Unmapped,46-50,178750.0000,FULLY DISBURSED,LAP,11-15 Yrs,LAP,LAP-RESALE INDEPENDENT HOUSE,2023-06-05,FULLY DISBURSED,0M_curr,stage 1,0,0M,LAP1ALL000074445 0M,NaN,1300000.0,NaN,NaN,NaN
1,LAP1ALL000074449,1352351,<15L,0.1375,15.50%-16.50%,2023-04-03,CRA,ALLEPPEY HL,ALLEPPEY HL,Alleppey Area,Kerala 2,South 1,ACTIVE,LAP,LT50%,FLOATING,AFFORDABLE,Unmapped,Unmapped,56-60,185948.2625,FULLY DISBURSED,LAP,11-15 Yrs,LAP,LAP-RESALE INDEPENDENT HOUSE,2023-06-05,FULLY DISBURSED,0M_curr,stage 1,0,0M,LAP1ALL000074449 0M,NaN,1352351.0,NaN,NaN,NaN
2,LAP1ALL000074450,650000,<15L,0.1375,15.50%-16.50%,2023-04-03,DSA,ALLEPPEY HL,ALLEPPEY HL,Alleppey Area,Kerala 2,South 1,ACTIVE,LAP,LT50%,FLOATING,AFFORDABLE,Unmapped,Unmapped,<35,89375.0000,FULLY DISBURSED,LAP,11-15 Yrs,LAP,LAP-RESALE INDEPENDENT HOUSE,2023-06-05,FULLY DISBURSED,0M_curr,stage 1,0,0M,LAP1ALL000074450 0M,NaN,650000.0,NaN,NaN,NaN


In [51]:
new= new.drop('DISB_STATUS', axis=1)
new.rename(columns={'DISBS_STATUS':'DISB_STATUS'}, inplace=True)
new.head()

,AGREEMENTNO,AMTFIN,LOAN BAND,BUSINESS_IRR,IRR BAND,AGR_AUTH_DATE,CHANNEL TYPE,BRANCHDESC,LOCATION,AREA,REGION,ZONE,STATUS_DESC,SCHEMEDESC,LTV BAND,PROPERTY_DESC,CONSTITUTION_DESC,INDUSTRYDESC,SUB INDUSTRYDESC,AGE BAND,Current WIRR,DISB_STATUS,PRODUCT1,Tenure Slab,End Use 1,Product_type,EMI_STARTDATE,Attribute,Value,mob_order,mob,conc,POS_N,DISBURSEDAMT,DPD,CUSTOMERNAME,month
0,LAP1ALL000074445,1300000,<15L,0.1375,15.50%-16.50%,2023-04-03,DSA,ALLEPPEY HL,ALLEPPEY HL,Alleppey Area,Kerala 2,South 1,ACTIVE,LAP,LT50%,FLOATING,AFFORDABLE,Unmapped,Unmapped,46-50,178750.0000,FULLY DISBURSED,LAP,11-15 Yrs,LAP,LAP-RESALE INDEPENDENT HOUSE,2023-06-05,0M_curr,stage 1,0,0M,LAP1ALL000074445 0M,NaN,1300000.0,NaN,NaN,NaN
1,LAP1ALL000074449,1352351,<15L,0.1375,15.50%-16.50%,2023-04-03,CRA,ALLEPPEY HL,ALLEPPEY HL,Alleppey Area,Kerala 2,South 1,ACTIVE,LAP,LT50%,FLOATING,AFFORDABLE,Unmapped,Unmapped,56-60,185948.2625,FULLY DISBURSED,LAP,11-15 Yrs,LAP,LAP-RESALE INDEPENDENT HOUSE,2023-06-05,0M_curr,stage 1,0,0M,LAP1ALL000074449 0M,NaN,1352351.0,NaN,NaN,NaN
2,LAP1ALL000074450,650000,<15L,0.1375,15.50%-16.50%,2023-04-03,DSA,ALLEPPEY HL,ALLEPPEY HL,Alleppey Area,Kerala 2,South 1,ACTIVE,LAP,LT50%,FLOATING,AFFORDABLE,Unmapped,Unmapped,<35,89375.0000,FULLY DISBURSED,LAP,11-15 Yrs,LAP,LAP-RESALE INDEPENDENT HOUSE,2023-06-05,0M_curr,stage 1,0,0M,LAP1ALL000074450 0M,NaN,650000.0,NaN,NaN,NaN
3,HL25ALL000074444,2800000,25L-30L,0.0850,13.50%-14.50%,2023-04-03,DSA,ALLEPPEY HL,ALLEPPEY HL,Alleppey Area,Kerala 2,South 1,ACTIVE,HL,70%-80%,FLOATING,AFFORDABLE,Unmapped,Unmapped,<35,238000.0000,FULLY DISBURSED,HL,16-20 Yrs,SELF CONSTRUCTION,RESIDENTIAL SELF CONSTRUCTION,2023-06-05,0M_curr,stage 1,0,0M,HL25ALL000074444 0M,NaN,2800000.0,NaN,NaN,NaN
4,LAP1ALL000074448,1083837,<15L,0.1375,15.50%-16.50%,2023-04-03,DSA,THIRUVALLA HL,THIRUVALLA HL,Alleppey Area,Kerala 2,South 1,ACTIVE,LAP,LT50%,FLOATING,AFFORDABLE,Unmapped,Unmapped,46-50,149027.5875,FULLY DISBURSED,LAP,11-15 Yrs,LAP,LAP-RESALE INDEPENDENT HOUSE,2023-06-05,0M_curr,stage 1,0,0M,LAP1ALL000074448 0M,NaN,1083837.0,NaN,NaN,NaN


In [52]:
new.columns

Index(['AGREEMENTNO', 'AMTFIN', 'LOAN BAND', 'BUSINESS_IRR', 'IRR BAND',
       'AGR_AUTH_DATE', 'CHANNEL TYPE', 'BRANCHDESC', 'LOCATION', 'AREA',
       'REGION', 'ZONE', 'STATUS_DESC', 'SCHEMEDESC', 'LTV BAND',
       'PROPERTY_DESC', 'CONSTITUTION_DESC', 'INDUSTRYDESC',
       'SUB INDUSTRYDESC', 'AGE BAND', 'Current WIRR', 'DISB_STATUS',
       'PRODUCT1', 'Tenure Slab', 'End Use 1', 'Product_type', 'EMI_STARTDATE',
       'Attribute', 'Value', 'mob_order', 'mob', 'conc', 'POS_N',
       'DISBURSEDAMT', 'DPD', 'CUSTOMERNAME', 'month'],
      dtype='object')

In [53]:
new.drop_duplicates().to_csv("D:\TASK\HL & LAP\HL\HL_static Pool\SPA wrt DM.csv", index=False)
print("saved successfully!!")

saved successfully!!
